# Exemplary script for network calculation of feed and reflux line network

- Creation of entire network from GIS data see file *networkModelling_example.py*

In [ ]:
### Imports
import os

import rasterio
import pandas as pd
from pathlib import Path
import geopandas as gp
import pandapipes as ppi
import numpy as np

from pandapipes.control import run_control
import geoppi

from geoppi import (transfer_LoadPoint_ppi, extract_FluidProperties_ppi, get_dict_from_aggregated_groups, implement_controllers, )

flp = Path(r"data/exampleNetwork")

In [ ]:
### Set target feed temperature (°C)
tfeed_winter = 85
treflux_winter = 60
tfeed_summer = 70
treflux_summer = 55

### Load data and define parameters

net = ppi.from_pickle(flp / "exampleNetwork_ppi.p")
profile = pd.read_csv(flp / Path(r"heat_demand_profile.txt"), sep = "\t", header=0).to_numpy().flatten()

tfeed = np.ones(8760) * (tfeed_winter + 273.15)
tfeed[3500:5500] = (tfeed_summer + 273.15)

treflux = np.ones(8760) * (treflux_winter + 273.15)
treflux[3500:5500] = (treflux_summer + 273.15)
deltaT = tfeed - treflux

net.heat_consumer["demand_use_th"] = net.heat_consumer["demand_use_th"].fillna(15000)

In [ ]:
net.heat_consumer["demand_use_th"]

In [ ]:
net.res_heat_consumer

In [ ]:
np.nanmin(net.res_junction["p_bar"].values)

In [ ]:
Qth_consumers

In [ ]:
### Define containers for results
Qth_producers = np.zeros((2, 8760))
Qth_consumers = np.zeros(8760)

Pel_pump = np.zeros((2, 8760)) # El. effort for pumping

pmin = np.zeros(8760) # Min. pressure in network

tfeed_consumers = np.zeros((len(net.heat_consumer), 8760))

## Further parameters
cp_fluid, rho_fluid, nu_fluid, g = extract_FluidProperties_ppi(net = net, t = 0.5 * (tfeed_winter + treflux_winter))


### Start calculation
np.seterr(all='ignore')
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for nn in range(100):
    # Set parameters
    net.heat_consumer["qext_w"] = net.heat_consumer["demand_use_th"] * profile[nn] * 1e03
    net.heat_consumer["controlled_mdot_kg_per_s"] = net.heat_consumer["qext_w"] / cp_fluid / deltaT[nn]

    net.controller.loc[0].object.target_T = treflux[nn]
    net.circ_pump_pressure["t_flow_k"] = tfeed[nn]

    run_control(net = net, max_iter = 25)

    # Write results
    tfeed_consumers[:,nn] = net.res_heat_consumer["t_from_k"].values
    pmin[nn] = np.nanmin(net.res_junction["p_bar"].values)

    Qth_producers[:, nn] = [
        (net.res_circ_pump_pressure["mdot_from_kg_per_s"].values * (net.res_circ_pump_pressure["t_to_k"] - net.res_circ_pump_pressure["t_from_k"]).values * cp_fluid)[0],
        (net.res_circ_pump_mass["mdot_from_kg_per_s"].values * (net.res_circ_pump_mass["t_to_k"] - net.res_circ_pump_mass["t_from_k"]).values * cp_fluid)[0]
    ]

    Qth_consumers[nn] = np.nansum(net.res_heat_consumer["mdot_from_kg_per_s"].values * (net.res_heat_consumer["t_to_k"] - net.res_heat_consumer["t_from_k"]).values * cp_fluid)



